# Optical Spectrum Classification & Analysis Pipeline
-------
## Overview
This notebook implements an automated pipeline for the classification and analysis of galaxy optical spectra. It distinguishes between various galaxy types—**Star-Forming (SF), Active Galactic Nuclei (AGN: Broad and narrow), LINER, Composite, and Passive**—using machine learning models trained on spectral features.

This notebook is tailored for applying the diagnostic to large catalogs that have already measuremnts for the EWs 

In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))
import pandas as pd
import numpy as np
import EW_classifier as ewc
import joblib

In [2]:
clf = joblib.load('../Models/Auto_OptSepcClassifier_SVM.sav')
scaler_sv = joblib.load('../Models/Auto_OptSepcClassifier_scaler.sav')

In [3]:
scaler_AGN_Type = joblib.load('../Models/Auto_OptSepcClassifier_scaler_AGN_type.sav')
clf_AGN_Type = joblib.load('../Models/Auto_OptSepcClassifier_SVM_AGN_type.sav')

In [4]:
path_to_AGN_spec = '../Example data/SDSS_example_spectrum_broad_line_AGN.csv'
path_to_SFG_spec = '../Example data/SDSS_example_spectrum_SFG.csv'

df_csv = pd.read_csv(f'{path_to_SFG_spec}')

flux = df_csv['flux']
wavelength = df_csv['wavelength']
var = df_csv['var']

In [5]:
columns = ['EW_NII_Ha', 'EW_OIII', 'EW_Hb', 'EW_NII_Ha_unc', 'EW_OIII_unc', 'EW_Hb_unc']

data = [
    [-97.55, -9.28, -10.18, 1.37, 0.43, 0.38],
    [-96.10, -9.05, -10.50, 1.40, 0.41, 0.39]
    ]

df_example = pd.DataFrame(data, columns=columns)

In [6]:
rows = []
for index, row in df_example.iterrows():
    label, means, stds = ewc.rf_classify_mc_EWs(row[['EW_NII_Ha', 'EW_OIII', 'EW_Hb']], row[['EW_NII_Ha_unc', 'EW_OIII_unc', 'EW_Hb_unc']], clf, scaler_sv)
    res = {**{'label': label}, **means.add_suffix('_mean').to_dict(), **stds.add_suffix('_std').to_dict()}
    rows.append(res)

df_results = pd.DataFrame(rows, index=df_example.index)
df_example = pd.concat([df_example, df_results], axis=1)
df_example

/Users/babis/OptSpecClassifier/EW_classifier.py:261: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  df['EW_NII_HA_NII_NON'] = np.random.normal(loc=EWs[0], scale=EWs_unc[0], size=n_mc)
/Users/babis/OptSpecClassifier/EW_classifier.py:262: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  df['OIII_5007_EQW_NON'] = np.random.normal(loc=EWs[1], scale=EWs_unc[1], size=n_mc)
/Users/babis/OptSpecClassifier/EW_classifier.py:263: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by pos

,EW_NII_Ha,EW_OIII,EW_Hb,EW_NII_Ha_unc,EW_OIII_unc,EW_Hb_unc,label,mc_proba_SF_mean,mc_proba_AGN_mean,mc_proba_LINER_mean,mc_proba_COMP_mean,mc_proba_PAS_mean,mc_proba_SF_std,mc_proba_AGN_std,mc_proba_LINER_std,mc_proba_COMP_std,mc_proba_PAS_std
0,-97.55,-9.28,-10.18,1.37,0.43,0.38,SF,0.965257,0.000251,0.000362,0.034053,0.000077,0.015934,0.000095,0.000135,0.015948,0.000016
1,-96.10,-9.05,-10.50,1.40,0.41,0.39,SF,0.982310,0.000176,0.000264,0.017170,0.000081,0.009478,0.000082,0.000139,0.009327,0.000018


In [ ]:
df_example.to_csv('path-to-save/classification_result.csv', index=False)

In [7]:
# EOF